In [ ]:
"""
PROGRESSION MODEL (TRIPLE ATTENTION + CDR) FOR DRISHTI-GS
=========================================================
Features:
- MobileNetV2 + MSCA + LBFR + PPM (Triple Attention)
- Focal EIoU Loss + Boundary Loss + MC Dropout
- Glaucoma scoring based on Vertical Cup-to-Disc Ratio (vCDR) in post-processing
- Adapted for merged Drishti-GS dataset
"""

import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import glob
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, backend as K
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import ModelCheckpoint, ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, confusion_matrix, accuracy_score
import albumentations as A
from PIL import Image

# ============================================================================
# UTILS & DATA LOADING (DRISHTI-GS ADAPTED)
# ============================================================================

def configure_gpu():
    print("Configuring GPU settings...")
    physical_devices = tf.config.list_physical_devices('GPU')
    if physical_devices:
        for device in physical_devices:
            tf.config.experimental.set_memory_growth(device, True)
        print(f"✓ GPU configured: {len(physical_devices)} device(s)")
    return bool(physical_devices)

def find_mask_for_image(img_path):
    """Find corresponding Cup/Disc masks for Drishti-GS"""
    img_dir = os.path.dirname(img_path)
    img_name = os.path.splitext(os.path.basename(img_path))[0]
    
    cup_path, disc_path = None, None
    
    # Search recursively in parent directories for GT
    search_root = os.path.dirname(os.path.dirname(img_dir))
    for root, _, files in os.walk(search_root):
        for f in files:
            if img_name in f:
                if 'cup' in f.lower() and ('seg' in f.lower() or 'map' in f.lower()):
                    cup_path = os.path.join(root, f)
                if ('od' in f.lower() or 'disc' in f.lower()) and ('seg' in f.lower() or 'map' in f.lower()):
                    disc_path = os.path.join(root, f)
    return cup_path, disc_path

def process_drishti_mask(cup_path, disc_path, target_size=(512,512)):
    """Combine masks into 0=Bg, 1=Disc, 2=Cup"""
    if not cup_path or not disc_path: return None
    try:
        cup = cv2.imread(cup_path, cv2.IMREAD_GRAYSCALE)
        disc = cv2.imread(disc_path, cv2.IMREAD_GRAYSCALE)
        if cup is None or disc is None: return None
        
        cup = cv2.resize(cup, target_size)
        disc = cv2.resize(disc, target_size)
        _, cup_bin = cv2.threshold(cup, 127, 255, cv2.THRESH_BINARY)
        _, disc_bin = cv2.threshold(disc, 127, 255, cv2.THRESH_BINARY)
        
        mask = np.zeros(target_size, dtype=np.uint8)
        mask[disc_bin > 0] = 1
        mask[cup_bin > 0] = 2
        return mask
    except: return None

def load_drishti_data(root_dir, img_size=(512,512)):
    """Recursively load Drishti-GS images and masks"""
    print(f"Scanning {root_dir}...")
    image_paths = []
    for root, _, files in os.walk(root_dir):
        for ext in ['*.png', '*.jpg']:
            for f in glob.glob(os.path.join(root, ext)):
                if not any(x in f.lower() for x in ['seg', 'map', 'gt']):
                    image_paths.append(f)
                    
    images, masks, names = [], [], []
    for img_path in image_paths:
        cup_p, disc_p = find_mask_for_image(img_path)
        if cup_p and disc_p:
            img = cv2.imread(img_path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, img_size) / 255.0
                mask = process_drishti_mask(cup_p, disc_p, img_size)
                if mask is not None:
                    images.append(img)
                    masks.append(mask)
                    names.append(os.path.basename(img_path))
                    
    print(f"Loaded {len(images)} images.")
    return np.array(images, dtype=np.float32), np.array(masks, dtype=np.uint8), names

def create_splits(images, masks, names):
    """60/20/20 Split"""
    ids = np.arange(len(images))
    # Test Split (20%)
    train_val_idx, test_idx = train_test_split(ids, test_size=0.2, random_state=42)
    # Val Split (Total 20%)
    train_idx, val_idx = train_test_split(train_val_idx, test_size=0.25, random_state=42)
    
    return (images[train_idx], masks[train_idx]), \
           (images[val_idx], masks[val_idx]), \
           (images[test_idx], masks[test_idx], [names[i] for i in test_idx])

# ============================================================================
# LOSS FUNCTIONS (Focal EIoU + Boundary)
# ============================================================================

def focal_loss(alpha=0.25, gamma=2.0):
    def loss(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, 1e-7, 1.0 - 1e-7)
        cross_entropy = -y_true * tf.math.log(y_pred)
        weight = alpha * y_true * ((1 - y_pred) ** gamma)
        return tf.reduce_sum(weight * cross_entropy, axis=-1)
    return loss

def eiou_loss(y_true, y_pred, smooth=1e-7):
    # Simplified IOU for stability
    y_true_f = K.flatten(y_true)
    y_pred_f = K.flatten(y_pred)
    intersection = K.sum(y_true_f * y_pred_f)
    union = K.sum(y_true_f) + K.sum(y_pred_f) - intersection
    iou = (intersection + smooth) / (union + smooth)
    return 1 - iou

def boundary_loss(y_true, y_pred):
    # Approximate implementation using pooling to find edges
    y_true_float = K.cast(y_true, 'float32')
    y_pred_float = K.cast(y_pred, 'float32')
    
    # Simple edge detection using AvgPooling (erosion)
    pooled_true = layers.AveragePooling2D(pool_size=(3, 3), strides=(1, 1), padding='same')(y_true_float)
    pooled_pred = layers.AveragePooling2D(pool_size=(3, 3), strides=(1, 1), padding='same')(y_pred_float)
    
    bound_true = K.abs(y_true_float - pooled_true)
    bound_pred = K.abs(y_pred_float - pooled_pred)
    
    return K.mean(K.square(bound_true - bound_pred))

def combined_loss(weights=[0.25, 1.0, 1.0]): # Alpha for classes 0, 1, 2
    focal = focal_loss(alpha=0.25)
    def loss(y_true, y_pred):
        l_focal = focal(y_true, y_pred)
        l_eiou = eiou_loss(y_true, y_pred)
        l_boundary = boundary_loss(y_true, y_pred)
        return l_focal + l_eiou + 0.5 * l_boundary
    return loss

# ============================================================================
# TRIPLE ATTENTION MODULES
# ============================================================================

class MSCA(layers.Layer):
    """Multi-Scale Context Attention"""
    def __init__(self, channels):
        super(MSCA, self).__init__()
        self.conv1_1 = layers.Conv2D(channels, 1, padding='same')
        self.conv5_5 = layers.Conv2D(channels, 5, padding='same', groups=channels)
        self.conv7_7 = layers.Conv2D(channels, 7, padding='same', groups=channels)
        self.conv11_11 = layers.Conv2D(channels, 11, padding='same', groups=channels)
        self.conv_out = layers.Conv2D(channels, 1, padding='same')
    
    def call(self, inputs):
        x = self.conv1_1(inputs)
        x5 = self.conv5_5(x)
        x7 = self.conv7_7(x)
        x11 = self.conv11_11(x)
        attn = x5 + x7 + x11
        attn = self.conv_out(attn)
        return inputs * attn

class LBFR(layers.Layer):
    """Local Boundary Feature Refinement"""
    def __init__(self, channels):
        super(LBFR, self).__init__()
        self.conv1 = layers.Conv2D(channels, 3, padding='same')
        self.conv2 = layers.Conv2D(channels, 1, padding='same')
        
    def call(self, inputs):
        x = self.conv1(inputs)
        return self.conv2(x) + inputs

class PPM(layers.Layer):
    """Pyramid Pooling Module"""
    def __init__(self, channels, bins=[1, 2, 3, 6]):
        super(PPM, self).__init__()
        self.bins = bins
        self.channels = channels
        
    def call(self, inputs):
        shape = tf.shape(inputs)[1:3]
        out_list = [inputs]
        for b in self.bins:
            pool = tf.reduce_mean(inputs, axis=[1, 2], keepdims=True)
            conv = layers.Conv2D(self.channels // len(self.bins), 1, padding='same')(pool)
            up = tf.image.resize(conv, shape)
            out_list.append(up)
        return layers.Concatenate()(out_list)

def mc_dropout_predict(model, X, n_samples=10): # Efficient implementation
    """Monte Carlo Dropout Prediction"""
    batch_size = X.shape[0]
    output_shape = model.output_shape[1:]
    
    # Initialize sum accumulator
    sum_predictions = np.zeros((batch_size,) + output_shape, dtype=np.float32)
    
    print(f"Running MC Dropout ({n_samples} samples)...")
    for i in range(n_samples):
        batch_preds = model.predict(X, batch_size=4, verbose=0)
        sum_predictions += batch_preds
        K.clear_session() # Clean up graph
        
    return sum_predictions / n_samples

# ============================================================================
# TRIPLE ATTENTION MODEL
# ============================================================================

def build_triple_attn_model(input_shape=(512,512,3), num_classes=3):
    inputs = layers.Input(input_shape)
    backbone = MobileNetV2(input_tensor=inputs, weights='imagenet', include_top=False)
    
    # Skips
    s1 = backbone.get_layer('block_1_expand_relu').output
    s2 = backbone.get_layer('block_3_expand_relu').output
    s3 = backbone.get_layer('block_6_expand_relu').output
    s4 = backbone.get_layer('block_13_expand_relu').output
    bridge = backbone.output
    
    # Attention on Bridge
    bridge = MSCA(1280)(bridge)
    bridge = PPM(1280)(bridge)
    bridge = layers.Dropout(0.3)(bridge) # For MC Dropout
    
    # Decoder
    x = layers.UpSampling2D((2,2))(bridge)
    x = layers.Concatenate()([x, s4])
    x = LBFR(512+96)(x) # Apply LBFR on features
    x = layers.Conv2D(512, 3, padding='same', activation='relu')(x)
    
    x = layers.UpSampling2D((2,2))(x)
    x = layers.Concatenate()([x, s3])
    x = LBFR(512+32)(x)
    x = layers.Conv2D(256, 3, padding='same', activation='relu')(x)
    
    x = layers.UpSampling2D((2,2))(x)
    x = layers.Concatenate()([x, s2])
    x = layers.Conv2D(128, 3, padding='same', activation='relu')(x)
    
    x = layers.UpSampling2D((2,2))(x)
    x = layers.Concatenate()([x, s1])
    x = layers.Conv2D(64, 3, padding='same', activation='relu')(x)
    
    x = layers.UpSampling2D((2,2))(x)
    outputs = layers.Conv2D(num_classes, 1, activation='softmax')(x)
    
    return models.Model(inputs, outputs, name="TripleAttn_Drishti")

# ============================================================================
# CDR & SCORING
# ============================================================================

def calculate_cdr_and_score(mask_pred):
    """
    Calculate Vertical Cup-to-Disc Ratio (vCDR) and Glaucoma Score
    Label 1 = Disc, Label 2 = Cup
    """
    disc_pixels = np.sum(mask_pred == 1) + np.sum(mask_pred == 2) # Cup is inside Disc
    cup_pixels = np.sum(mask_pred == 2)
    
    if disc_pixels == 0: return 0, "Healthy"
    
    # Vertical diameter estimation (bounding box height)
    rows_disc = np.any((mask_pred==1)|(mask_pred==2), axis=1)
    rows_cup = np.any(mask_pred==2, axis=1)
    
    h_disc = np.sum(rows_disc)
    h_cup = np.sum(rows_cup)
    
    if h_disc == 0: return 0, "Healthy"
    
    vcdr = h_cup / h_disc
    
    # Basic Glaucoma Scoring Rule (Refuge/Drishti standard)
    # vCDR > 0.6 is typically suspicious/glaucomatous
    label = "Glaucoma" if vcdr > 0.6 else "Healthy"
    
    return vcdr, label

# ============================================================================
# MAIN PIPELINE
# ============================================================================

def custom_generator(images, masks, batch_size):
    """Simple generator"""
    while True:
        idx = np.random.permutation(len(images))
        for i in range(0, len(images), batch_size):
            batch_idx = idx[i:i+batch_size]
            bx = images[batch_idx]
            by = masks[batch_idx]
            by_cat = to_categorical(by, num_classes=3)
            yield bx, by_cat

# Main Execution
if __name__ == "__main__":
    configure_gpu()
    root_dir = "/kaggle/input/datasets/ayush02102001/glaucoma-classification-datasets/DRISHTI-GS/DRISHTI-GS/"
    
    # Load Data
    images, masks, names = load_drishti_data(root_dir)
    
    if len(images) > 0:
        (X_train, y_train), (X_val, y_val), (X_test, y_test, names_test) = create_splits(images, masks, names)
        
        # Train
        model = build_triple_attn_model()
        model.compile(optimizer='adam', loss=combined_loss(), metrics=['accuracy'])
        
        # Callbacks
        ckpt = ModelCheckpoint("best_triple_drishti.keras", save_best_only=True, monitor='val_loss')
        
        # Fit
        model.fit(
            custom_generator(X_train, y_train, 4),
            steps_per_epoch=len(X_train)//4,
            validation_data=custom_generator(X_val, y_val, 4),
            validation_steps=len(X_val)//4,
            epochs=100,
            callbacks=[ckpt]
        )
        
        # Predict with MC Dropout
        preds_mc = mc_dropout_predict(model, X_test, n_samples=5)
        preds_labels = np.argmax(preds_mc, axis=-1)
        
        # Calculate Scores
        results = []
        for i, pred_mask in enumerate(preds_labels):
            vcdr, status = calculate_cdr_and_score(pred_mask)
            results.append({
                "Image":names_test[i],
                "vCDR": vcdr,
                "Prediction": status
            })
            
            # Save Mask
            vis_mask = np.zeros_like(pred_mask, dtype=np.uint8)
            vis_mask[pred_mask==1] = 127
            vis_mask[pred_mask==2] = 255
            Image.fromarray(vis_mask).save(f"results/test_{i}_mask.png")
            
        # Save CSV
        df = pd.DataFrame(results)
        df.to_csv("drishti_glaucoma_scores.csv", index=False)
        print("Scoring Complete. Saved to drishti_glaucoma_scores.csv")
    else:
        print("Data load failed.")